# Road Surface Type Classification Task

## Phase 1

### Section 1: List of requirements

This section records the software environment and package versions required to reproduce the dataset audit and Phase 1 exploratory data analysis on the JupyterHub VM.

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.helper.inventory import get_environment_info

env_info = get_environment_info()
pd.DataFrame(list(env_info["packages"].items()), columns=["Package", "Version"])

,Package,Version
0,numpy,2.3.5
1,pandas,3.0.6
2,opencv-python,not installed
3,pillow,12.3.0


### Section 2: Description of the dataset

StreetSurfaceVis contains 9,122 crowdsourced street-level images collected through Mapillary from locations within Germany. Each image is manually labeled for surface type and surface quality ([Kapp et al., 2025](https://doi.org/10.1038/s41597-024-04295-9)). 

Our task predicts surface type across five classes: asphalt, concrete, paving stones, sett, and unpaved.

In [2]:
import os

BASE_DATA = ROOT_DIR / "data"
METADATA_PATH = str(BASE_DATA / "streetSurfaceVis_v1_0.csv")
DATA_PATH = str(BASE_DATA / "s_256")

### Section 3: Exploratory data analysis

We check every CSV record for repeated image IDs and fully repeated rows. These counts describe different problems: an ID can recur even if its other fields differ.


In [3]:
from src.helper.inventory import load_and_validate_metadata

metadata_df, metadata_summary = load_and_validate_metadata(METADATA_PATH)
pd.DataFrame([metadata_summary]).T.rename(columns={0: "Value"})


,Value
total_records,9122
expected_total,9122
train_records,8346
expected_train,8346
test_records,776
expected_test,776
unique_ids,9122
duplicate_ids,0
fully_repeated_rows,0
has_nulls,False


We check whether each named image file exists and can be opened. Only readable files enter the byte comparison below.


In [4]:
from src.helper.inventory import audit_image_inventory

inventory_df, audit_summary, missing_ids, decode_fails, extra_files = (
    audit_image_inventory(metadata_df, DATA_PATH)
)
pd.DataFrame([audit_summary]).T.rename(columns={0: "Count"})

,Count
expected_images,9122
found_images,9122
readable_images,9120
missing_images,0
decode_failures,2
extra_files,0
unusable_images,2


We inspect the decode failures identified during the image audit, displaying their image identifiers, failure reasons, and corresponding metadata records.

In [5]:
decode_df = pd.DataFrame(
    decode_fails, columns=["mapillary_image_id", "decode_error", "file_size_bytes"]
)
metadata_df[metadata_df["mapillary_image_id"].isin(decode_df["mapillary_image_id"])].merge(
    decode_df, on="mapillary_image_id"
)

,mapillary_image_id,user_id,user_name,captured_at,longitude,latitude,train,surface_type,surface_quality,decode_error,file_size_bytes
0,3865590250223952,103078715264357,hotteliene,1583081300392,10.581377,53.325355,True,concrete,excellent,cannot identify image file '/home/jupyter-csc9...,0
1,2289217127888486,106289114939007,goobersmash,1461932090212,13.035185,52.404632,True,sett,good,cannot identify image file '/home/jupyter-csc9...,0


We check for any missing image files or extra untracked files in the image directory to ensure complete dataset alignment.

In [6]:
file_discrepancies = {
    "missing_image_ids": missing_ids,
    "extra_filenames": extra_files,
}
pd.DataFrame([file_discrepancies]).T.rename(columns={0: "Discrepancies"})

,Discrepancies
missing_image_ids,[]
extra_filenames,[]


The audit reconciles 9,122 expected records with 9,120 readable images, 2 decode failures of size 0 bytes, and 0 missing or extra files. The verified inventory dataframe is preserved for subsequent split and balance checks.

We compare the bytes of every distinct readable file in the 256-pixel release. Files with the same digest are checked byte for byte before they count as identical. The summary counts groups within train, within test, and across the published split; a larger group can count in more than one category.


In [7]:
from src.helper.inventory import audit_duplicate_images

duplicate_summary, duplicate_groups, read_failures = audit_duplicate_images(inventory_df)
assert duplicate_summary["compared_files"] + duplicate_summary["read_failures"] == inventory_df.loc[inventory_df["is_readable"], "file_path"].nunique()
pd.DataFrame([duplicate_summary]).T.rename(columns={0: "Count"})


,Count
compared_files,9120
read_failures,0
identical_file_groups,8
affected_image_ids,16
within_train_groups,8
within_test_groups,0
cross_split_groups,0
conflicting_label_groups,1


Each row below gives an affected image ID, both labels, and its published train/test flag. Group numbers tie identical files together. An empty table means no identical readable files were found.


In [8]:
duplicate_groups[["group", "mapillary_image_id", "surface_type", "surface_quality", "train"]]


,group,mapillary_image_id,surface_type,surface_quality,train
0,1,155881879847389,paving_stones,good,True
1,1,216839193329808,paving_stones,good,True
2,2,1823349551191443,sett,bad,True
3,2,1889128151294434,sett,bad,True
4,3,255475163307416,paving_stones,intermediate,True
5,3,992905331431472,paving_stones,intermediate,True
6,4,321048266708127,sett,bad,True
7,4,956168328603371,sett,bad,True
8,5,324818063042590,sett,intermediate,True
9,5,439906807819690,sett,intermediate,True


The full CSV has 0 repeated image IDs and 0 fully repeated rows. Of 9,122 named files, 9,120 were readable; 2 zero-byte files could not be opened and none were missing. Eight identical-file groups affect 16 IDs. All eight are within the published training partition; none are within test or across train/test. One group has conflicting surface-quality labels.


### Section 4: Dataset Split

The published train/test assignments stay unchanged. Keep each identical training group together when assigning train and validation. Set aside groups with conflicting labels or members on both sides of the published split for team review. For an agreeing cross-split group, the proposed approach is to retain its test member and exclude training copies. Review within-test repeats when deciding final evaluation counts. Issue #14 will decide inclusion. Exact byte matching cannot find different photos of the same road; the CSV has no road or image-sequence ID. Missing and unreadable files could not be compared.


The handoff lists every affected image ID with a proposed action. The conflicting-label group needs a team decision before its files enter modeling.


In [9]:
handoff = []
for number, rows in duplicate_groups.groupby("group"):
    ids = ", ".join(rows["mapillary_image_id"])
    flags = set(rows["train"])
    labels = rows[["surface_type", "surface_quality"]].drop_duplicates()
    if len(labels) > 1:
        action = "Set aside for team review: conflicting labels"
    elif flags == {True, False}:
        action = "Review split; retain test member, exclude training copies"
    elif flags == {True}:
        action = "Keep together in train/validation; decide copy count in #14"
    else:
        action = "Review repeated test content in #14"
    handoff.append({"group": number, "image_ids": ids, "proposed_handling": action})
pd.DataFrame(handoff, columns=["group", "image_ids", "proposed_handling"])


,group,image_ids,proposed_handling
0,1,"155881879847389, 216839193329808",Keep together in train/validation; decide copy...
1,2,"1823349551191443, 1889128151294434",Keep together in train/validation; decide copy...
2,3,"255475163307416, 992905331431472",Keep together in train/validation; decide copy...
3,4,"321048266708127, 956168328603371",Keep together in train/validation; decide copy...
4,5,"324818063042590, 439906807819690",Keep together in train/validation; decide copy...
5,6,"343903797624675, 429598338923343",Keep together in train/validation; decide copy...
6,7,"481593120303132, 643661443549586",Keep together in train/validation; decide copy...
7,8,"529834985029767, 690122179011682",Set aside for team review: conflicting labels


### Section 5: Traditional ML model training

## 